In [2]:
pip install torch torchaudio transformers espnet speechbrain

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 3070, in _dep_map
    return self.__dep_map
  File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 2863, in __getattr__
    raise AttributeError(attr)
AttributeError: _DistInfoDistribution__dep_map

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 3061, in _parsed_pkg_info
    return self._pkg_info
  File "/usr/local/lib/python3.10/dist-packages/pip/_vendor/pkg_resources/__init__.py", line 2863, in __getattr__
    raise AttributeError(attr)
AttributeError: _pkg_info. Did you mean: 'egg_info'?

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py",

In [4]:
import os
import random
import shutil
import torchaudio

# Define paths
clips_folder = "/content/drive/MyDrive/clips"
output_folder = "/content/sample_data/output_data"
train_folder = os.path.join(output_folder, "train")
val_folder = os.path.join(output_folder, "validation")
test_folder = os.path.join(output_folder, "test")

# Create output directories
os.makedirs(train_folder, exist_ok=True)
os.makedirs(val_folder, exist_ok=True)
os.makedirs(test_folder, exist_ok=True)

# Function to preprocess audio
def preprocess_audio(file_path, target_sample_rate=16000):
    waveform, sample_rate = torchaudio.load(file_path)
    if sample_rate != target_sample_rate:
        resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
        waveform = resampler(waveform)
    return waveform

# Split dataset (80% train, 10% val, 10% test)
def split_dataset(clips_folder):
    all_files = [f for f in os.listdir(clips_folder) if f.endswith(".mp3")]
    random.shuffle(all_files)
    train_split = int(0.8 * len(all_files))
    val_split = int(0.9 * len(all_files))
    train_files = all_files[:train_split]
    val_files = all_files[train_split:val_split]
    test_files = all_files[val_split:]
    return train_files, val_files, test_files

# Process and save files
def process_and_save(files, source_folder, target_folder):
    for file in files:
        source_path = os.path.join(source_folder, file)
        target_path = os.path.join(target_folder, file)
        waveform = preprocess_audio(source_path)
        torchaudio.save(target_path, waveform, 16000)

# Split and process
train_files, val_files, test_files = split_dataset(clips_folder)
process_and_save(train_files, clips_folder, train_folder)
process_and_save(val_files, clips_folder, val_folder)
process_and_save(test_files, clips_folder, test_folder)

print("Preprocessing and splitting completed!")


Preprocessing and splitting completed!


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
import torchaudio
import torchaudio.transforms as T
import torch
import torch.nn.functional as F

# Define a function to extract Mel-spectrogram features
def extract_features(file_path, target_sample_rate=16000, n_mels=64, target_length=800):
    waveform, sample_rate = torchaudio.load(file_path)

    # Resample if necessary
    if sample_rate != target_sample_rate:
        resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
        waveform = resampler(waveform)

    # Compute the Mel-spectrogram
    mel_spec_transform = T.MelSpectrogram(sample_rate=target_sample_rate, n_mels=n_mels)
    mel_spec = mel_spec_transform(waveform)

    # Optionally, apply Log-Mel spectrogram
    log_mel_spec = T.AmplitudeToDB()(mel_spec)

    # Add batch dimension (batch_size = 1 here, as we are processing one file at a time)
    log_mel_spec = log_mel_spec.unsqueeze(0)  # Shape: (1, channels, sequence_length)

    # Pad or truncate the sequence to the target length
    current_length = log_mel_spec.size(2)

    if current_length < target_length:
        # Pad the sequence (pad at the end)
        padding = target_length - current_length
        log_mel_spec = F.pad(log_mel_spec, (0, padding))  # Pad the sequence length dimension
    elif current_length > target_length:
        # Truncate the sequence
        log_mel_spec = log_mel_spec[:, :, :target_length]  # Keep only the first 'target_length' frames

    return log_mel_spec

# Define the folder path and target output folder
clips_folder = "/content/drive/MyDrive/clips"
output_folder = "/content/sample_data/output_data/Output_Features"

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Set the target length for padding/truncation
target_length = 800  # Adjust this value as needed

# Loop over all .mp3 files in the clips folder and extract features
for filename in os.listdir(clips_folder):
    if filename.endswith(".mp3"):
        file_path = os.path.join(clips_folder, filename)

        # Extract features
        features = extract_features(file_path, target_length=target_length)

        # Save the features (you can save them as .pt files)
        output_file_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}.pt")
        torch.save(features, output_file_path)
        print(f"Features saved for {filename}")

print("Feature extraction completed for all files!")


Features saved for common_voice_ur_26652455.mp3
Features saved for common_voice_ur_26703775.mp3
Features saved for common_voice_ur_27859453.mp3
Features saved for common_voice_ur_26653137.mp3
Features saved for common_voice_ur_26756862.mp3
Features saved for common_voice_ur_26630226.mp3
Features saved for common_voice_ur_26652629.mp3
Features saved for common_voice_ur_26652703.mp3
Features saved for common_voice_ur_26652001.mp3
Features saved for common_voice_ur_27849055.mp3
Features saved for common_voice_ur_26704005.mp3
Features saved for common_voice_ur_27466070.mp3
Features saved for common_voice_ur_27804564.mp3
Features saved for common_voice_ur_26651951.mp3
Features saved for common_voice_ur_27847471.mp3
Features saved for common_voice_ur_26653328.mp3
Features saved for common_voice_ur_27849052.mp3
Features saved for common_voice_ur_27804265.mp3
Features saved for common_voice_ur_27465571.mp3
Features saved for common_voice_ur_26643286.mp3
Features saved for common_voice_ur_27859

DEFINE MODEL

In [7]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T


# Define the feature extraction function
def extract_features(file_path, target_sample_rate=16000, n_mels=64, target_length=800):
    waveform, sample_rate = torchaudio.load(file_path)

    # Resample if necessary
    if sample_rate != target_sample_rate:
        resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
        waveform = resampler(waveform)

    # Compute the Mel-spectrogram
    mel_spec_transform = T.MelSpectrogram(sample_rate=target_sample_rate, n_mels=n_mels)
    mel_spec = mel_spec_transform(waveform)

    # Apply Log-Mel spectrogram
    log_mel_spec = T.AmplitudeToDB()(mel_spec)

    # Pad or truncate the sequence to the target length
    current_length = log_mel_spec.size(2)
    if current_length < target_length:
        padding = target_length - current_length
        log_mel_spec = F.pad(log_mel_spec, (0, padding))  # Pad the sequence length dimension
    elif current_length > target_length:
        log_mel_spec = log_mel_spec[:, :, :target_length]  # Keep only the first 'target_length' frames

    return log_mel_spec


class ConformerMoE(nn.Module):
    def __init__(self, input_dim, num_classes, num_experts=4, hidden_dim=128):
        super(ConformerMoE, self).__init__()
        self.num_experts = num_experts
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim

        # Define the Conformer layer (Conv1d layer)
        self.conformer_layer = nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1)

        # Define the Mixture of Experts (MoE) layers
        self.experts = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(num_experts)])

        # Define a gating network for selecting experts
        self.gate = nn.Linear(hidden_dim, num_experts)

        # Output layer for classification
        self.output_layer = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # Input shape: (batch_size, sequence_length, input_dim)
        print(f"Input shape: {x.shape}")

        # Permute to (batch_size, input_dim, sequence_length) for Conv1d
        x = x.permute(0, 2, 1)  # Shape: (batch_size, input_dim, sequence_length)
        print(f"Shape after permutation: {x.shape}")

        # Pass through Conv1d layer
        x = self.conformer_layer(x)  # Shape: (batch_size, hidden_dim, sequence_length)
        print(f"Shape after Conv1d: {x.shape}")

        # Apply activation function
        x = F.relu(x)

        # Flatten for gating
        batch_size, hidden_dim, seq_len = x.size()
        x_flatten = x.permute(0, 2, 1).reshape(-1, hidden_dim)  # Shape: (batch_size * seq_len, hidden_dim)
        print(f"Shape after flattening for gating: {x_flatten.shape}")

        # Gating mechanism
        gate_output = self.gate(x_flatten)  # Shape: (batch_size * seq_len, num_experts)
        gate_output = F.softmax(gate_output, dim=-1)

        # Process through experts
        expert_outputs = [expert(x_flatten) for expert in self.experts]  # List of (batch_size * seq_len, hidden_dim)
        expert_outputs = torch.stack(expert_outputs, dim=1)  # Shape: (batch_size * seq_len, num_experts, hidden_dim)

        # Multiply gating weights with expert outputs
        gate_output = gate_output.unsqueeze(-1)  # Shape: (batch_size * seq_len, num_experts, 1)
        expert_output = torch.sum(gate_output * expert_outputs, dim=1)  # Shape: (batch_size * seq_len, hidden_dim)

        # Reshape back to original dimensions
        expert_output = expert_output.view(batch_size, seq_len, hidden_dim)  # Shape: (batch_size, seq_len, hidden_dim)
        print(f"Shape after expert processing: {expert_output.shape}")

        # Pass through the output layer
        output = self.output_layer(expert_output)  # Shape: (batch_size, seq_len, num_classes)
        print(f"Output shape: {output.shape}")

        return output
# Dataset class for loading features
class SpeechDataset(Dataset):
    def __init__(self, features_path, target_length=800):
        self.features_path = features_path
        self.files = os.listdir(features_path)
        self.target_length = target_length  # Target length for sequence

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_name = self.files[idx]
        file_path = os.path.join(self.features_path, file_name)

        try:
            features = torch.load(file_path)
            features = features.squeeze(0).squeeze(0).permute(1, 0).unsqueeze(0)

            # Generate dummy targets with the correct shape
            targets = torch.randint(0, 10, (1, self.target_length))  # Correct shape for CTC Loss
            return features, targets
        except Exception as e:
            print(f"Error loading features for file {file_path}: {e}")
            return None, None

# Collate function for DataLoader
def collate_fn(batch):
    features, targets = zip(*batch)
    max_length = max([f.size(1) for f in features])

    padded_features = []
    for feature in features:
        padding_length = max_length - feature.size(1)
        padding = torch.zeros(1, padding_length, feature.size(2))
        padded_feature = torch.cat((feature, padding), dim=1)
        padded_features.append(padded_feature)

    padded_features = torch.stack(padded_features, dim=0)
    targets = torch.stack(targets, dim=0)

    return padded_features, targets

input_dim = 64  # Adjust according to the feature size from your Mel spectrogram
num_classes = 10  # Adjust according to the number of output classes
model = ConformerMoE(input_dim=input_dim, num_classes=num_classes)

# Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Create the dataset and dataloader
features_path = '/content/sample_data/output_data/Output_Features'  # Replace with actual path
train_dataset = SpeechDataset(features_path)
train_loader = DataLoader(train_dataset, batch_size=32, collate_fn=collate_fn)

# Training loop with fixes
for epoch in range(10):
    model.train()
    for batch_idx, (features, targets) in enumerate(train_loader):
        optimizer.zero_grad()

        # Squeeze and adjust feature dimensions
        features = features.squeeze(1)  # Remove unnecessary dimensions
        print(f"Feature shape: {features.shape}")

        # Forward pass
        output = model(features)  # Shape: (batch_size, seq_len, num_classes)
        print(f"Model output shape: {output.shape}")

        # Transpose output for CTC Loss: (T, N, C)
        output = output.permute(1, 0, 2)  # Shape: (time_steps, batch_size, num_classes)

        # Apply log_softmax for log probabilities
        log_probs = F.log_softmax(output, dim=2)

        # Calculate target lengths before flattening
        target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)

        # Ensure targets is a 2D tensor of shape (batch_size, target_length)
        targets = targets.squeeze(1)  # Shape: (batch_size, target_length)
        print(f"Target shape after squeezing: {targets.shape}")

        # Define input lengths
        input_lengths = torch.full((log_probs.size(1),), log_probs.size(0), dtype=torch.long)  # All same sequence length

        try:
            # Compute CTC Loss
            loss = nn.CTCLoss()(log_probs, targets, input_lengths, target_lengths)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            # Print progress
            if batch_idx % 10 == 0:
                print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item()}")

        except RuntimeError as e:
            print(f"Error in loss calculation: {e}")


<ipython-input-7-6fe3852b944d>:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  features = torch.load(file_path)


Streaming output truncated to the last 5000 lines.
Model output shape: torch.Size([32, 1344, 10])
Target shape after squeezing: torch.Size([32, 800])
Feature shape: torch.Size([32, 1768, 64])
Input shape: torch.Size([32, 1768, 64])
Shape after permutation: torch.Size([32, 64, 1768])
Shape after Conv1d: torch.Size([32, 128, 1768])
Shape after flattening for gating: torch.Size([56576, 128])
Shape after expert processing: torch.Size([32, 1768, 128])
Output shape: torch.Size([32, 1768, 10])
Model output shape: torch.Size([32, 1768, 10])
Target shape after squeezing: torch.Size([32, 800])
Feature shape: torch.Size([32, 1350, 64])
Input shape: torch.Size([32, 1350, 64])
Shape after permutation: torch.Size([32, 64, 1350])
Shape after Conv1d: torch.Size([32, 128, 1350])
Shape after flattening for gating: torch.Size([43200, 128])
Shape after expert processing: torch.Size([32, 1350, 128])
Output shape: torch.Size([32, 1350, 10])
Model output shape: torch.Size([32, 1350, 10])
Target shape after s

In [8]:
# Save the model
model_path = '/content/sample_data/conformer_moe_model.pth'  # Specify the path where you want to save the model
torch.save(model.state_dict(), model_path)

In [16]:
import torch
from torch.utils.data import Dataset, DataLoader
import os
import torchaudio
from torchaudio.transforms import Resample
from pathlib import Path

# Ensure that torchaudio uses the ffmpeg backend
torchaudio.set_audio_backend("ffmpeg")

def extract_features(file_path, target_sample_rate=16000, n_mels=64, target_length=800):
    waveform, sample_rate = torchaudio.load(file_path)

    # Resample if necessary
    if sample_rate != target_sample_rate:
        resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
        waveform = resampler(waveform)

    # Compute the Mel-spectrogram
    mel_spec_transform = T.MelSpectrogram(sample_rate=target_sample_rate, n_mels=n_mels)
    mel_spec = mel_spec_transform(waveform)

    # Apply Log-Mel spectrogram
    log_mel_spec = T.AmplitudeToDB()(mel_spec)

    # Pad or truncate the sequence to the target length
    current_length = log_mel_spec.size(2)
    if current_length < target_length:
        padding = target_length - current_length
        log_mel_spec = F.pad(log_mel_spec, (0, padding))  # Pad the sequence length dimension
    elif current_length > target_length:
        log_mel_spec = log_mel_spec[:, :, :target_length]  # Keep only the first 'target_length' frames

    return log_mel_spec

# Dataset class for loading features (same strategy you used)
class InferenceDataset(Dataset):
    def __init__(self, clips_folder, target_length=800):
        self.clips_folder = clips_folder
        self.clips = [f for f in os.listdir(clips_folder) if f.endswith('.mp3')]  # Only MP3 files
        self.target_length = target_length

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip_path = os.path.join(self.clips_folder, self.clips[idx])
        features = extract_features(clip_path, target_length=self.target_length)
        return features

# Inference function (same as before)
def infer(model, dataloader, device):
    model.eval()
    all_predictions = []

    with torch.no_grad():
        # Inside the inference loop
        for features in dataloader:
            features = features.squeeze(1).to(device)
            features = features.permute(0, 2, 1)

            # Output shape and values
            output = model(features)  # Shape: (batch_size, seq_len, num_classes)

            _, predicted = output.max(2)  # Get the predicted labels

            all_predictions.append(predicted)


    return all_predictions

# Load the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ConformerMoE(input_dim=64, num_classes=10)  # Ensure these match your model's parameters
model.load_state_dict(torch.load('/content/sample_data/conformer_moe_model.pth'))  # Path to your .pth model file
model.to(device)

# Prepare the data
clips_folder = '/content/drive/MyDrive/clips'  # Replace with the actual path to your clips folder
inference_dataset = InferenceDataset(clips_folder)
inference_loader = DataLoader(inference_dataset, batch_size=1, shuffle=False)

# Run inference
predictions = infer(model, inference_loader, device)


<ipython-input-16-697076938743>:9: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("ffmpeg")
<ipython-input-16-697076938743>:75: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case 

Streaming output truncated to the last 5000 lines.
Shape after expert processing: torch.Size([1, 800, 128])
Output shape: torch.Size([1, 800, 10])
Input shape: torch.Size([1, 800, 64])
Shape after permutation: torch.Size([1, 64, 800])
Shape after Conv1d: torch.Size([1, 128, 800])
Shape after flattening for gating: torch.Size([800, 128])
Shape after expert processing: torch.Size([1, 800, 128])
Output shape: torch.Size([1, 800, 10])
Input shape: torch.Size([1, 800, 64])
Shape after permutation: torch.Size([1, 64, 800])
Shape after Conv1d: torch.Size([1, 128, 800])
Shape after flattening for gating: torch.Size([800, 128])
Shape after expert processing: torch.Size([1, 800, 128])
Output shape: torch.Size([1, 800, 10])
Input shape: torch.Size([1, 800, 64])
Shape after permutation: torch.Size([1, 64, 800])
Shape after Conv1d: torch.Size([1, 128, 800])
Shape after flattening for gating: torch.Size([800, 128])
Shape after expert processing: torch.Size([1, 800, 128])
Output shape: torch.Size([1,

Average Overall Loss

In [21]:
# Training loop with overall model loss calculation
for epoch in range(10):
    model.train()
    epoch_loss = 0  # Accumulator for overall epoch loss
    for batch_idx, (features, targets) in enumerate(train_loader):
        optimizer.zero_grad()

        features = features.squeeze(1)  # Remove unnecessary dimensions
        output = model(features)  # Shape: (batch_size, seq_len, num_classes)

        output = output.permute(1, 0, 2)  # Shape: (time_steps, batch_size, num_classes)
        log_probs = F.log_softmax(output, dim=2)

        target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
        targets = targets.squeeze(1)

        input_lengths = torch.full((log_probs.size(1),), log_probs.size(0), dtype=torch.long)

        try:
            loss = nn.CTCLoss()(log_probs, targets, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()

            # Accumulate loss for the epoch
            epoch_loss += loss.item()

            if batch_idx % 10 == 0:
                print(f"Epoch {epoch}, Batch {batch_idx}, Loss: {loss.item()}")

        except RuntimeError as e:
            print(f"Error in loss calculation: {e}")

    # Calculate the average loss for the entire epoch
    avg_epoch_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch} - Average Loss: {avg_epoch_loss}")


<ipython-input-19-b030c9f2cfb5>:114: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  features = torch.load(file_path)


Streaming output truncated to the last 5000 lines.
Output shape: torch.Size([4, 1169, 50])
Epoch 8, Batch 250, Loss: 3.2687489986419678
Input shape: torch.Size([4, 1056, 64])
Shape after permutation: torch.Size([4, 64, 1056])
Shape after Conv1d: torch.Size([4, 128, 1056])
Shape after flattening for gating: torch.Size([4224, 128])
Shape after expert processing: torch.Size([4, 1056, 128])
Output shape: torch.Size([4, 1056, 50])
Input shape: torch.Size([4, 1768, 64])
Shape after permutation: torch.Size([4, 64, 1768])
Shape after Conv1d: torch.Size([4, 128, 1768])
Shape after flattening for gating: torch.Size([7072, 128])
Shape after expert processing: torch.Size([4, 1768, 128])
Output shape: torch.Size([4, 1768, 50])
Input shape: torch.Size([4, 1284, 64])
Shape after permutation: torch.Size([4, 64, 1284])
Shape after Conv1d: torch.Size([4, 128, 1284])
Shape after flattening for gating: torch.Size([5136, 128])
Shape after expert processing: torch.Size([4, 1284, 128])
Output shape: torch.Si

In [ ]:
#with transcript to train model
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T


# Define the feature extraction function
def extract_features(file_path, target_sample_rate=16000, n_mels=64, target_length=800):
    waveform, sample_rate = torchaudio.load(file_path)

    # Resample if necessary
    if sample_rate != target_sample_rate:
        resampler = T.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)
        waveform = resampler(waveform)

    # Compute the Mel-spectrogram
    mel_spec_transform = T.MelSpectrogram(sample_rate=target_sample_rate, n_mels=n_mels)
    mel_spec = mel_spec_transform(waveform)

    # Apply Log-Mel spectrogram
    log_mel_spec = T.AmplitudeToDB()(mel_spec)

    # Pad or truncate the sequence to the target length
    current_length = log_mel_spec.size(2)
    if current_length < target_length:
        padding = target_length - current_length
        log_mel_spec = F.pad(log_mel_spec, (0, padding))  # Pad the sequence length dimension
    elif current_length > target_length:
        log_mel_spec = log_mel_spec[:, :, :target_length]  # Keep only the first 'target_length' frames

    return log_mel_spec


# MoE-based Conformer Model for Speech-to-Text
class ConformerMoE(nn.Module):
    def __init__(self, input_dim, num_classes, num_experts=4, hidden_dim=128):
        super(ConformerMoE, self).__init__()
        self.num_experts = num_experts
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim

        # Define the Conformer layer (Conv1d layer)
        self.conformer_layer = nn.Conv1d(input_dim, hidden_dim, kernel_size=3, padding=1)

        # Define the Mixture of Experts (MoE) layers
        self.experts = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(num_experts)])

        # Define a gating network for selecting experts
        self.gate = nn.Linear(hidden_dim, num_experts)

        # Output layer for classification
        self.output_layer = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # Input shape: (batch_size, sequence_length, input_dim)
        x = x.permute(0, 2, 1)  # Shape: (batch_size, input_dim, sequence_length)

        # Pass through Conv1d layer
        x = self.conformer_layer(x)  # Shape: (batch_size, hidden_dim, sequence_length)

        # Apply activation function
        x = F.relu(x)

        # Flatten for gating
        batch_size, hidden_dim, seq_len = x.size()
        x_flatten = x.permute(0, 2, 1).reshape(-1, hidden_dim)  # Shape: (batch_size * seq_len, hidden_dim)

        # Gating mechanism
        gate_output = self.gate(x_flatten)  # Shape: (batch_size * seq_len, num_experts)
        gate_output = F.softmax(gate_output, dim=-1)

        # Process through experts
        expert_outputs = [expert(x_flatten) for expert in self.experts]  # List of (batch_size * seq_len, hidden_dim)
        expert_outputs = torch.stack(expert_outputs, dim=1)  # Shape: (batch_size * seq_len, num_experts, hidden_dim)

        # Multiply gating weights with expert outputs
        gate_output = gate_output.unsqueeze(-1)  # Shape: (batch_size * seq_len, num_experts, 1)
        expert_output = torch.sum(gate_output * expert_outputs, dim=1)  # Shape: (batch_size * seq_len, hidden_dim)

        # Reshape back to original dimensions
        expert_output = expert_output.view(batch_size, seq_len, hidden_dim)  # Shape: (batch_size, seq_len, hidden_dim)

        # Pass through the output layer
        output = self.output_layer(expert_output)  # Shape: (batch_size, seq_len, num_classes)

        return output


# Dataset class for loading features with transcriptions
class SpeechDataset(Dataset):
    def __init__(self, features_path, transcriptions_path, target_length=800):
        self.features_path = features_path
        self.transcriptions_path = transcriptions_path
        self.files = os.listdir(features_path)
        self.target_length = target_length  # Target length for sequence
        self.transcriptions = self._load_transcriptions()

    def _load_transcriptions(self):
        # Load the transcriptions from a file (this assumes a one-to-one correspondence with features)
        with open(self.transcriptions_path, 'r') as f:
            transcriptions = f.readlines()
        return transcriptions

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_name = self.files[idx]
        file_path = os.path.join(self.features_path, file_name)
        transcription = self.transcriptions[idx].strip()  # Get the transcription corresponding to the file

        try:
            features = torch.load(file_path)
            features = features.squeeze(0).squeeze(0).permute(1, 0).unsqueeze(0)

            # Convert transcription to indices (you can use a dictionary for characters -> indices)
            targets = torch.tensor([ord(c) for c in transcription], dtype=torch.long)  # Just an example of converting characters
            return features, targets
        except Exception as e:
            print(f"Error loading features for file {file_path}: {e}")
            return None, None


# Collate function for DataLoader
def collate_fn(batch):
    features, targets = zip(*batch)
    max_length = max([f.size(1) for f in features])

    padded_features = []
    for feature in features:
        padding_length = max_length - feature.size(1)
        padding = torch.zeros(1, padding_length, feature.size(2))
        padded_feature = torch.cat((feature, padding), dim=1)
        padded_features.append(padded_feature)

    padded_features = torch.stack(padded_features, dim=0)
    targets = torch.stack(targets, dim=0)

    return padded_features, targets


# Training configuration
input_dim = 64  # Adjust according to the feature size from your Mel spectrogram
num_classes = 256  # Adjust to the size of your character set or vocabulary
num_experts = 4  # Number of experts in MoE
hidden_dim = 128  # Hidden layer size for MoE
target_length = 800  # Target length for the sequence

# Create model, dataset, and dataloader
model = ConformerMoE(input_dim=input_dim, num_classes=num_classes, num_experts=num_experts, hidden_dim=hidden_dim)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

features_path = '/path/to/features'  # Replace with actual path to extracted features
transcriptions_path = '/path/to/transcriptions.txt'  # Replace with actual path to transcription data
train_dataset = SpeechDataset(features_path, transcriptions_path)
train_loader = DataLoader(train_dataset, batch_size=32, collate_fn=collate_fn)

# Training loop
for epoch in range(10):
    model.train()
    for batch_idx, (features, targets) in enumerate(train_loader):
        optimizer.zero_grad()

        # Squeeze and adjust feature dimensions
        features = features.squeeze(1)  # Remove unnecessary dimensions

        # Forward pass
        output = model(features)  # Shape: (batch_size, seq_len, num_classes)

        # Transpose output for CTC Loss: (T, N, C)
        output = output.permute(1, 0, 2)  # Shape: (time_steps, batch_size, num_classes)

        # Apply log_softmax for log probabilities
        log_probs = F.log_softmax(output, dim=2)

        # Calculate target lengths before flattening
        target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)

        # Ensure targets is a 2D tensor of shape (batch_size, target_length)
        targets = targets.squeeze(1)  # Shape: (batch_size, target_length)

        # Define input lengths
        input_lengths = torch.full((log_probs.size(1),), log_probs.size(0), dtype=torch.long)  # All same sequence length

        try:
            # Compute CTC Loss
            loss = nn.CTCLoss()(log_probs, targets, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()
            print(f"Epoch {epoch + 1}, Batch {batch_idx + 1}, Loss: {loss.item():.4f}")
        except Exception as e:
            print(f"Error during training batch {batch_idx}: {e}")

WHISPER MODEL

In [7]:
pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.54.4
    Uninstalling openai-1.54.4:
      Successfully uninstalled openai-1.54.4


In [18]:
!pip install -U openai-whisper
!pip install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.5/800.5 kB 14.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.5/209.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.2 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20240930-py3-none-any.whl size=803320 sha256=7683981072b5a5179356413581e664b79640d898fe744ae8e2d7d64ec7dcec88
  Stored in directory: /root/.cache/pip/wheels/dd/4a/1f/d1c4bf3b9133c8168fe617ed979cab7b14fe381d059ffb9d83
Successfully built openai-whisper
  Preparing metadata (setup.py) ... done
  Created wheel for ffmpeg: filename=ffmpeg-1.4-py3-none-any.whl size=6082 sha256=767f72c5496a58261db566e028c66f5ed1c3ca7ff432625cd3705bf1f7a3804d
  Stored in directory: /root/.cache/pip/wheels/8e/7a/69/cd6aeb83b126a7f04cbe7c9d929028dc52a6e7d525ff56003a
Successfully built 

In [ ]:
import whisper
import os

# Load the Whisper model
model = whisper.load_model("large")  # You can choose other sizes: tiny, small, medium, large

# Directory containing audio files
audio_folder_path = "/content/drive/MyDrive/clips"

# Function to transcribe audio
def transcribe_audio(file_path):
    try:
        print(f"Processing file: {file_path}")
        result = model.transcribe(file_path)
        return result.get("text", "")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Process all audio files in the folder
for file_name in os.listdir(audio_folder_path):
    if file_name.endswith((".mp3", ".wav")):
        file_path = os.path.join(audio_folder_path, file_name)
        transcription = transcribe_audio(file_path)
        if transcription:
            print(f"Transcription for {file_name}:\n{transcription}\n")


100%|█████████████████████████████████████| 2.88G/2.88G [00:39<00:00, 78.9MiB/s]
/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this exper

Processing file: /content/drive/MyDrive/clips/common_voice_ur_26652455.mp3
Transcription for common_voice_ur_26652455.mp3:
 اس کی جمع کو ایک سا کر سکہ ہے یا رنگ کے تفافت میں اضافہ کر سکے ہیں

Processing file: /content/drive/MyDrive/clips/common_voice_ur_26703775.mp3
Transcription for common_voice_ur_26703775.mp3:
 یہ بیس عدب کے بارے میں بھی اٹھی تھی

Processing file: /content/drive/MyDrive/clips/common_voice_ur_27859453.mp3
Transcription for common_voice_ur_27859453.mp3:
 اب نواز شریف کس پر الزام دھریں گے؟

Processing file: /content/drive/MyDrive/clips/common_voice_ur_26653137.mp3
Transcription for common_voice_ur_26653137.mp3:
 اب صورتحال جائے کہ نواز شریف گئے تو ایک اور سٹیٹس کو کا

Processing file: /content/drive/MyDrive/clips/common_voice_ur_26756862.mp3
Transcription for common_voice_ur_26756862.mp3:
 الحمرہ میں معروف سرائی کی صحافی

Processing file: /content/drive/MyDrive/clips/common_voice_ur_26630226.mp3
Transcription for common_voice_ur_26630226.mp3:
 قومی الیکشن اسی سال مئی م